# Step 5 — Final Evaluation on the TEST Set
**Oral Diseases Image Classification · Sprint Two**

This notebook opens the **test set for the first and only time**. Rules:
- All five models are evaluated on the **same 500-ish test images** they have never seen (not for training, not for early stopping, not for choosing hyperparameters).
- **We do not pick or re-tune anything based on these numbers.** The model for the app was chosen on validation (tuned ResNet50). Test results are reported as they are, good or bad.
- Because the test set is small, every number comes with a **95% bootstrap confidence interval**, and the tuned ResNet50 is compared to each other model with a **paired bootstrap** (same resampled images for both models).
- We also show **validation → test**, so you can see how optimistic the validation numbers were.

**Models it looks for** (by file name, anywhere under `/kaggle/input` or `/kaggle/working`):
| Model | File | Comes from |
|---|---|---|
| Custom CNN | `custom_cnn_best.keras` | Step 2 notebook output |
| ResNet50 / EfficientNetB0 / MobileNetV2 | `<name>_best.keras` | Step 3 notebook output |
| ResNet50 (tuned) | `ResNet50_tuned_best.keras` | Step 4 notebook output |

**Before running**
1. *Settings → Accelerator → GPU T4 x2* (Internet can stay off).
2. *Add Input → Notebooks → Your Work →* add the **Step 1, 2, 3 and 4** notebooks (each must have a saved **Commit** version that contains its output files). If a model's file lives somewhere else, you can also upload it as a Dataset and add that.
3. Run All. The next cell tells you exactly which files were found and which are missing; missing models are skipped, nothing crashes.

In [ ]:
import os, json, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, accuracy_score

SEED, BATCH, N_BOOT = 42, 32, 2000
ROOTS    = [os.environ.get("DATA_ROOT", "/kaggle/input"), os.environ.get("WORK_DIR", "/kaggle/working")]
WORK_DIR = os.environ.get("WORK_DIR", "/kaggle/working")
os.makedirs(os.path.join(WORK_DIR, "results"), exist_ok=True)
REFERENCE = "ResNet50 (tuned)"          # the model chosen on VALIDATION; fixed before looking at test

EXPECTED = {                             # display name -> (model file, validation json written by Steps 2-4)
    "Custom CNN":       ("custom_cnn_best.keras",        "custom_cnn_val.json"),
    "ResNet50":         ("ResNet50_best.keras",          "ResNet50_val.json"),
    "EfficientNetB0":   ("EfficientNetB0_best.keras",    "EfficientNetB0_val.json"),
    "MobileNetV2":      ("MobileNetV2_best.keras",       "MobileNetV2_val.json"),
    "ResNet50 (tuned)": ("ResNet50_tuned_best.keras",    "ResNet50_tuned_val.json"),
}

def find_file(fname):
    for root in ROOTS:
        for d, _, files in os.walk(root):
            if fname in files:
                return os.path.join(d, fname)
    return None

# clean data (test split)
DATA_DIR = None
for root in ROOTS:
    for d, _, files in os.walk(root):
        if "manifest.csv" in files and "meta.json" in files and os.path.isdir(os.path.join(d, "test")):
            DATA_DIR = d; break
    if DATA_DIR: break
assert DATA_DIR, "Step-1 output (with the test folder) not found. Add the Step 1 notebook as input."
CLASS_NAMES = json.load(open(os.path.join(DATA_DIR, "meta.json")))["class_names"]
TEST_DIR = os.path.join(DATA_DIR, "test")

print("GPUs     :", tf.config.list_physical_devices("GPU") or "none (it will still work, just slower)")
print("DATA_DIR :", DATA_DIR)
print("Classes  :", CLASS_NAMES, "\n")

FOUND, VAL_JSON = {}, {}
rows = []
for name, (mf, jf) in EXPECTED.items():
    p, j = find_file(mf), find_file(jf)
    if p: FOUND[name] = p
    if j: VAL_JSON[name] = json.load(open(j))
    rows.append({"model": name, "model file": "FOUND" if p else "MISSING", "val json": "found" if j else "-",
                 "size_MB": round(os.path.getsize(p) / 1e6, 1) if p else None})
print(pd.DataFrame(rows).to_string(index=False))
missing = [n for n in EXPECTED if n not in FOUND]
if missing: print("\n!! Missing (will be skipped):", missing, "-> add the notebook that produced them as an Input (it needs a saved Commit).")

## 1) Load every model and predict on the test set

In [ ]:
@keras.utils.register_keras_serializable(package="oral")
class ImageNetPreprocess(layers.Layer):
    def __init__(self, mode="none", **kwargs):
        super().__init__(**kwargs); self.mode = mode
    def call(self, x):
        if self.mode == "none":
            return x
        return keras.applications.imagenet_utils.preprocess_input(x, mode=self.mode)
    def get_config(self):
        cfg = super().get_config(); cfg.update(mode=self.mode); return cfg

def test_ds(size):
    return keras.utils.image_dataset_from_directory(TEST_DIR, label_mode="categorical", class_names=CLASS_NAMES,
                                                    image_size=size, batch_size=BATCH, shuffle=False).prefetch(tf.data.AUTOTUNE)

Y, PRED, PROBS, INFO = None, {}, {}, {}
for name, path in FOUND.items():
    try:
        model = keras.models.load_model(path, compile=False)
    except Exception as e:
        print(f"!! could not load {name}: {type(e).__name__}: {str(e)[:150]}"); continue
    size = tuple(model.input_shape[1:3])
    ds = test_ds(size)
    y = np.concatenate([b.numpy().argmax(1) for _, b in ds])
    if Y is None: Y = y
    assert np.array_equal(Y, y), "test images/labels differ between models - this should never happen"
    p = model.predict(ds, verbose=0)
    t0 = time.time(); model.predict(ds, verbose=0); ms = (time.time() - t0) / len(y) * 1000
    PROBS[name], PRED[name] = p, p.argmax(1)
    INFO[name] = {"input": f"{size[0]}x{size[1]}", "params_M": round(model.count_params() / 1e6, 2),
                  "file_MB": round(os.path.getsize(path) / 1e6, 1), "ms_per_image(GPU,batched)": round(ms, 2)}
    print(f"{name:18s} loaded | input {size} | test accuracy {accuracy_score(Y, PRED[name]):.3f}")
    del model
print("\nTest images:", len(Y), "| per class:", dict(zip(CLASS_NAMES, np.bincount(Y, minlength=len(CLASS_NAMES)).tolist())))

## 2) Test results with 95% confidence intervals (bootstrap)

In [ ]:
K = len(CLASS_NAMES); n = len(Y)
rng = np.random.default_rng(SEED)
IDX = rng.integers(0, n, size=(N_BOOT, n))                 # the SAME resampled images are used for every model

def macro_f1_from(yb, pb):
    cm = np.bincount(yb * K + pb, minlength=K * K).reshape(K, K).astype(float)
    tp = np.diag(cm); prec = np.divide(tp, cm.sum(0), out=np.zeros(K), where=cm.sum(0) > 0)
    rec = np.divide(tp, cm.sum(1), out=np.zeros(K), where=cm.sum(1) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(K), where=(prec + rec) > 0)
    return f1.mean()

BOOT_F1, BOOT_ACC = {}, {}
for name, p in PRED.items():
    BOOT_F1[name]  = np.array([macro_f1_from(Y[ix], p[ix]) for ix in IDX])
    BOOT_ACC[name] = np.array([(Y[ix] == p[ix]).mean() for ix in IDX])

def ci(a): lo, hi = np.percentile(a, [2.5, 97.5]); return f"[{lo:.3f} - {hi:.3f}]"
res = []
for name, p in PRED.items():
    r = {"model": name, "test_accuracy": accuracy_score(Y, p), "acc_95%CI": ci(BOOT_ACC[name]),
         "test_macro_F1": f1_score(Y, p, average="macro"), "F1_95%CI": ci(BOOT_F1[name])}
    if name in VAL_JSON:
        r["val_macro_F1"] = VAL_JSON[name]["val_macro_f1"]; r["val->test F1 change"] = r["test_macro_F1"] - r["val_macro_F1"]
    res.append(r)
RES = pd.DataFrame(res).sort_values("test_macro_F1", ascending=False)
print(RES.round(3).to_string(index=False))
print("\n(The CI says: with only ~500 test images, the true score could plausibly be anywhere inside the bracket. Near-duplicate photos inside the test set are correlated, so the real uncertainty is a little larger than the bracket shows.)")

In [ ]:
if REFERENCE in PRED and len(PRED) > 1:
    print(f"Paired bootstrap: is {REFERENCE} really better than each other model? (macro-F1 difference, same test images)\n")
    out = []
    for name in PRED:
        if name == REFERENCE: continue
        d = BOOT_F1[REFERENCE] - BOOT_F1[name]
        lo, hi = np.percentile(d, [2.5, 97.5])
        out.append({"vs": name, "mean_diff": d.mean(), "95%CI_of_diff": f"[{lo:+.3f}, {hi:+.3f}]",
                    "P(reference better)": (d > 0).mean(), "verdict": "clearly better" if lo > 0 else ("clearly worse" if hi < 0 else "not distinguishable")})
    print(pd.DataFrame(out).round(3).to_string(index=False))
else:
    print(f"{REFERENCE} (or other models) not available - skipping paired comparison.")

## 3) Where does each model fail? (per-class recall and confusion matrices)

In [ ]:
recall = pd.DataFrame({name: dict(zip(CLASS_NAMES, (confusion_matrix(Y, p, labels=range(K)).diagonal() /
                                                    np.maximum(np.bincount(Y, minlength=K), 1)))) for name, p in PRED.items()})
recall.insert(0, "test_images", np.bincount(Y, minlength=K))
print("Per-class RECALL on test (share of each class that was found):")
print(recall.round(3).to_string())

names = list(PRED)
cols = 3; rows_ = int(np.ceil(len(names) / cols))
fig, axes = plt.subplots(rows_, cols, figsize=(6 * cols, 5.2 * rows_)); axes = np.atleast_1d(axes).ravel()
for ax, name in zip(axes, names):
    ConfusionMatrixDisplay(confusion_matrix(Y, PRED[name], labels=range(K)), display_labels=CLASS_NAMES).plot(ax=ax, xticks_rotation=45, colorbar=False)
    ax.set_title(f"{name}  (acc {accuracy_score(Y, PRED[name]):.3f})")
for ax in axes[len(names):]: ax.axis("off")
plt.tight_layout(); plt.show()

if REFERENCE in PRED:
    cm = confusion_matrix(Y, PRED[REFERENCE], labels=range(K))
    pairs = sorted([(CLASS_NAMES[i], CLASS_NAMES[j], int(cm[i, j])) for i in range(K) for j in range(K) if i != j and cm[i, j] > 0], key=lambda t: -t[2])
    print(f"\n{REFERENCE} - most frequent mistakes on TEST (true -> predicted: count):")
    for t, pr, c in pairs[:8]: print(f"  {t:14s} -> {pr:14s}: {c}")
    print("\n" + classification_report(Y, PRED[REFERENCE], labels=range(K), target_names=CLASS_NAMES, digits=3, zero_division=0))

## 4) Practical comparison (accuracy vs size vs speed)

In [ ]:
prac = RES[["model", "test_accuracy", "test_macro_F1"]].merge(pd.DataFrame(INFO).T.reset_index().rename(columns={"index": "model"}), on="model")
print(prac.round(3).to_string(index=False))
print("\nNote: ms/image was measured on the Kaggle GPU with batches of 32. On a normal CPU inside the app it will be slower (the app only handles one image at a time, so this is fine).")

fig, ax = plt.subplots(figsize=(8, 3.8))
order = RES.sort_values("test_macro_F1")
lo = [np.percentile(BOOT_F1[m], 2.5) for m in order.model]; hi = [np.percentile(BOOT_F1[m], 97.5) for m in order.model]
ax.barh(order.model, order.test_macro_F1, xerr=[order.test_macro_F1 - lo, np.array(hi) - order.test_macro_F1], color="#4C78A8", capsize=4)
ax.set_xlabel("test macro-F1 (bars = 95% CI)"); ax.set_xlim(max(0, min(lo) - 0.05), 1)
plt.tight_layout(); plt.show()

RES.to_csv(os.path.join(WORK_DIR, "results", "final_test_results.csv"), index=False)
json.dump({n: {"test_accuracy": float(accuracy_score(Y, p)), "test_macro_f1": float(f1_score(Y, p, average="macro"))} for n, p in PRED.items()},
          open(os.path.join(WORK_DIR, "results", "final_test_results.json"), "w"), indent=2)
print("Saved: results/final_test_results.csv / .json")